# Studying cases where members do not grow

In [1]:
import reframed
import cobra
import pandas as pd
import numpy as np
from os import listdir
import warnings
import json

In [2]:
import sys

sys.path.append("../functions/")
from minimal_medium import minimal_medium

In [3]:
%load_ext autoreload
%autoreload 2

*E. coli mutants*

In [4]:
models_ecoli = {}
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    for path in listdir("../models/Guillen2021/strains/"):
        if not path.endswith(".xml"): continue
        if path[:-4] not in ["mutant_aroC","mutant_thrA","mutant_metE"]:continue
        
        
        models_ecoli[path.split("_")[1][:-4]] = reframed.load_cbmodel("../models/Guillen2021/strains/"+path)

*For E. coli mutants (LB medium to see auxotrophies)*

In [6]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = list(media_db[media_db.medium=="LB"].compound.values)
compounds.append("btn")
# compounds.append("chor") - didn't have any effect...
env_ecoli = reframed.Environment.from_compounds(compounds)

In [11]:
# Compounds from study - these get priority (preferred reactions)
rxns = env_ecoli.keys()

missing_total = []
specific_solutions = []
sol_count_dfs = {}

# Find minimal medium for each model
for model_id,model in models_ecoli.items():
    print(model_id)
    
    # Find complete medium (based on reactions) for the model
    env_complete = reframed.Environment.complete(model)
    env_complete.apply(model,inplace=True,exclusive=True,warning=False)
    rxns_complete = env_complete.keys()
    
    # ...exclude specific reactions
    rxns_complete = list(set([e for e in rxns_complete if ("R_DM" not in e) and ("_acald_" not in e)]))
    
    # Find minimal medium
    solutions = minimal_medium(model,exchange_reactions=list(rxns_complete),preferred_reactions=rxns,min_growth=0.1,n_solutions=1000,use_pool=True,min_mass_weight=True)
    
    for sol in solutions[1]:
        print(sol)
        break  
    # Count occurence of a metabolite in minimal medium (across 1000 solutions) 
    sol_count = dict()
    for sol in solutions[0]:
        for rxn in sol:
            if rxn in sol_count.keys():
                sol_count[rxn]+=1
            else:
                sol_count[rxn]=1
    sol_count_df = pd.Series(sol_count).sort_values(ascending=False)
    sol_count_dfs[model_id]=sol_count_df
    
    # Include the compounds that are present in >500 solutions in the extended medium (missing_total)
    missing = list((set(sol_count_df[sol_count_df>500].index)-set(env_ecoli.keys())))
    print(missing)
    missing_total.extend(missing)

add2lb = list(set(missing_total))

thrA
Objective: 19950.02204902838
Status: Unknown

['R_EX_26dap__M_e']
metE
Objective: 899.12654344479
Status: Unknown

['R_EX_mmet_e']
aroC
[]


/var/folders/mc/1rm74nkd4s78l46281nyt5wm0000gn/T/ipykernel_23877/2231967669.py:34: DeprecationWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  sol_count_df = pd.Series(sol_count).sort_values(ascending=False)


In [10]:
missing_total

['R_EX_26dap__M_e', 'R_EX_mmet_e']

In [38]:
models_ecoli["metE"].metabolites.M_26dap__M_e

Meso-2,6-Diaminoheptanedioate

In [31]:
for rxn in models_ecoli["metE"].get_metabolite_reactions("M_26dap__M_c"):
    print(models_ecoli["metE"].reactions[rxn])
    genes = models_ecoli["metE"].reactions[rxn].get_genes()
    
    for gene in genes:
        print(models_ecoli["metE"].genes[gene])

R_DAPE: M_26dap_LL_c <-> M_26dap__M_c
dapF
R_UAAGDS: M_26dap__M_c + M_atp_c + M_uamag_c --> M_adp_c + M_h_c + M_pi_c + M_ugmd_c
murE
R_DAPDC: M_26dap__M_c + M_h_c --> M_co2_c + M_lys__L_c
lysA
R_DAPabcpp: M_26dap__M_p + M_atp_c + M_h2o_c --> M_26dap__M_c + M_adp_c + M_h_c + M_pi_c
R_LADGMDH: M_LalaDgluMdap_c + M_h2o_c --> M_26dap__M_c + M_LalaDglu_c
mpaA


In [39]:
models_ecoli["metE"].metabolites.M_mmet_e

S-Methyl-L-methionine

In [40]:
for rxn in models_ecoli["metE"].gene_to_reaction_lookup()["G_b0002"]:
    print(models_ecoli["metE"].reactions[rxn])

R_ASPK: M_asp__L_c + M_atp_c <-> M_4pasp_c + M_adp_c
R_HSDy: M_hom__L_c + M_nadp_c <-> M_aspsa_c + M_h_c + M_nadph_c


In [43]:
gene_id2name = pd.read_csv("../data/Guillen2021/generated/gene_id2name.tsv",sep="\t")

In [47]:
gene_id2name[gene_id2name.name.isin(["metH","metE"])]

,id,name
967,G_b3829,metE
971,G_b4019,metH


In [49]:
for rxn in models_ecoli["metE"].gene_to_reaction_lookup()["G_b4019"]:
    print(models_ecoli["metE"].reactions[rxn])

In [50]:
for rxn in models_ecoli["metE"].gene_to_reaction_lookup()["G_b3829"]:
    print(models_ecoli["metE"].reactions[rxn])

R_METS: M_5mthf_c + M_hcys__L_c --> M_h_c + M_met__L_c + M_thf_c [0.0, 0.0]
